In [ ]:

import pandas as pd

from google.colab import files

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    average_precision_score
)

print("Upload your CLASS A CSV:")
uploaded_a = files.upload()

class_a_file = list(uploaded_a.keys())[0]
class_a = pd.read_csv(class_a_file)

print("\nClass A columns:")
print(class_a.columns.tolist())

import kagglehub
import os

path = kagglehub.dataset_download(
    "colinmorris/reddit-usernames"
)

print("Dataset downloaded to:")
print(path)

print("\nFiles in dataset:")
print(os.listdir(path))


csv_files = [
    file for file in os.listdir(path)
    if file.endswith(".csv")
]

if not csv_files:
    raise ValueError("No CSV file was found in the downloaded Kaggle dataset.")

print("CSV files found:")
print(csv_files)

class_c_file = os.path.join(path, csv_files[0])
class_c = pd.read_csv(class_c_file)

print("\nClass C columns:")
print(class_c.columns.tolist())

print("\nNumber of Class C rows:")
print(len(class_c))


def find_username_column(df):

    possible_names = [
        "username",
        "user_name",
        "handle",
        "user",
        "author",
        "name"
    ]

    columns_lower = {
        str(col).lower().strip(): col
        for col in df.columns
    }

    for name in possible_names:
        if name in columns_lower:
            return columns_lower[name]

    raise ValueError(
        "Could not automatically find the username column. "
        f"Columns found: {df.columns.tolist()}"
    )


a_username_col = find_username_column(class_a)
c_username_col = find_username_column(class_c)

print("Class A username column:", a_username_col)
print("Class C username column:", c_username_col)


class_a_clean = class_a[[a_username_col]].copy()
class_c_clean = class_c[[c_username_col]].copy()

class_a_clean.columns = ["username"]
class_c_clean.columns = ["username"]

class_a_clean = class_a_clean.dropna()
class_c_clean = class_c_clean.dropna()

class_a_clean["username"] = class_a_clean["username"].astype(str)
class_c_clean["username"] = class_c_clean["username"].astype(str)

class_a_clean = class_a_clean.drop_duplicates("username")
class_c_clean = class_c_clean.drop_duplicates("username")

class_a_clean["label"] = 1
class_c_clean["label"] = 0

print("Class A usernames:", len(class_a_clean))
print("Class C usernames:", len(class_c_clean))

sample_size = 300

if len(class_c_clean) < sample_size:
    raise ValueError(
        f"Class C only contains {len(class_c_clean):,} unique usernames, "
        f"which is fewer than the requested {sample_size:,}."
    )

class_c_sample = class_c_clean.sample(
    n=sample_size,
    random_state=42
)

data = pd.concat(
    [class_a_clean, class_c_sample],
    ignore_index=True
)

# Shuffle rows
data = data.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

print("\nClass counts:")
print(data["label"].value_counts())

data.head()

X = data["username"]
y = data["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training usernames:", len(X_train))
print("Testing usernames:", len(X_test))
print("\nTraining class counts:")
print(y_train.value_counts())
print("\nTesting class counts:")
print(y_test.value_counts())

vectorizer = TfidfVectorizer(
    analyzer="char",
    ngram_range=(2, 5),
    lowercase=True
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print("Number of TF-IDF features:", X_train_tfidf.shape[1])


model = LogisticRegression(
    max_iter=1000,
    random_state=42,
    class_weight="balanced"
)

model.fit(X_train_tfidf, y_train)

print("Model trained successfully.")


predictions = model.predict(X_test_tfidf)
probabilities = model.predict_proba(X_test_tfidf)[:, 1]

print("Accuracy:")
print(accuracy_score(y_test, predictions))

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        predictions,
        target_names=["Class C", "Class A"]
    )
)

print("Confusion Matrix:")
print(confusion_matrix(y_test, predictions))

print("\nPR-AUC / Average Precision:")
print(average_precision_score(y_test, probabilities))


def classify_username(username):
    username_vector = vectorizer.transform([username])

    prediction = model.predict(username_vector)[0]
    probability = model.predict_proba(username_vector)[0][1]

    if prediction == 1:
        result = "Class A-like"
    else:
        result = "Class C-like"

    print("Username:", username)
    print("Prediction:", result)
    print("Class A probability:", round(probability, 3))



Upload your CLASS A CSV:


Saving Class_A_Criminal_Usernames_Master_Database.csv to Class_A_Criminal_Usernames_Master_Database (4).csv

Class A columns:
['Real Name', 'Username', 'Age', 'Sex', 'Nationality', 'Crime', 'Source', 'Source URL', 'Review Status', 'Notes']
Using Colab cache for faster access to the 'reddit-usernames' dataset.
Dataset downloaded to:
/kaggle/input/reddit-usernames

Files in dataset:
['users.csv']
CSV files found:
['users.csv']

Class C columns:
['author', 'n']

Number of Class C rows:
25865740
Class A username column: Username
Class C username column: author
Class A usernames: 234
Class C usernames: 25865739

Class counts:
label
0    300
1    234
Name: count, dtype: int64
Training usernames: 427
Testing usernames: 107

Training class counts:
label
0    240
1    187
Name: count, dtype: int64

Testing class counts:
label
0    60
1    47
Name: count, dtype: int64
Number of TF-IDF features: 7967
Model trained successfully.
Accuracy:
0.5420560747663551

Classification Report:
              pr

In [ ]:
classify_username("Shady11")

Username: Shady11
Prediction: Class C-like
Class A probability: 0.454


In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score
)

import numpy as np


X = data["username"]
y = data["label"]


skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)


accuracy_scores = []
precision_scores = []
recall_scores = []
f1_scores = []
pr_auc_scores = []


fold = 1


for train_index, test_index in skf.split(X, y):

    X_train = X.iloc[train_index]
    X_test = X.iloc[test_index]

    y_train = y.iloc[train_index]
    y_test = y.iloc[test_index]

    vectorizer = TfidfVectorizer(
        analyzer="char",
        ngram_range=(2, 5),
        lowercase=True
    )


    X_train_tfidf = vectorizer.fit_transform(X_train)
    X_test_tfidf = vectorizer.transform(X_test)

    model = LogisticRegression(
        max_iter=1000,
        random_state=42,
        class_weight="balanced"
    )

    model.fit(
        X_train_tfidf,
        y_train
    )

    predictions = model.predict(X_test_tfidf)

    probabilities = model.predict_proba(
        X_test_tfidf
    )[:, 1]

    accuracy = accuracy_score(
        y_test,
        predictions
    )

    precision = precision_score(
        y_test,
        predictions
    )

    recall = recall_score(
        y_test,
        predictions
    )

    f1 = f1_score(
        y_test,
        predictions
    )

    pr_auc = average_precision_score(
        y_test,
        probabilities
    )


    accuracy_scores.append(accuracy)
    precision_scores.append(precision)
    recall_scores.append(recall)
    f1_scores.append(f1)
    pr_auc_scores.append(pr_auc)


    print(f"\nFOLD {fold}")
    print("--------------------")
    print("Accuracy:", round(accuracy, 3))
    print("Class A Precision:", round(precision, 3))
    print("Class A Recall:", round(recall, 3))
    print("Class A F1:", round(f1, 3))
    print("PR-AUC:", round(pr_auc, 3))

    fold += 1


print("5-FOLD CROSS-VALIDATION RESULTS")

print(
    "Accuracy:",
    round(np.mean(accuracy_scores), 3),
    "+/-",
    round(np.std(accuracy_scores), 3)
)

print(
    "Class A Precision:",
    round(np.mean(precision_scores), 3),
    "+/-",
    round(np.std(precision_scores), 3)
)

print(
    "Class A Recall:",
    round(np.mean(recall_scores), 3),
    "+/-",
    round(np.std(recall_scores), 3)
)

print(
    "Class A F1:",
    round(np.mean(f1_scores), 3),
    "+/-",
    round(np.std(f1_scores), 3)
)

print(
    "PR-AUC:",
    round(np.mean(pr_auc_scores), 3),
    "+/-",
    round(np.std(pr_auc_scores), 3)
)


FOLD 1
--------------------
Accuracy: 0.598
Class A Precision: 0.559
Class A Recall: 0.404
Class A F1: 0.469
PR-AUC: 0.595

FOLD 2
--------------------
Accuracy: 0.626
Class A Precision: 0.6
Class A Recall: 0.447
Class A F1: 0.512
PR-AUC: 0.623

FOLD 3
--------------------
Accuracy: 0.664
Class A Precision: 0.649
Class A Recall: 0.511
Class A F1: 0.571
PR-AUC: 0.736

FOLD 4
--------------------
Accuracy: 0.579
Class A Precision: 0.521
Class A Recall: 0.532
Class A F1: 0.526
PR-AUC: 0.546

FOLD 5
--------------------
Accuracy: 0.585
Class A Precision: 0.538
Class A Recall: 0.304
Class A F1: 0.389
PR-AUC: 0.566

5-FOLD CROSS-VALIDATION RESULTS
Accuracy: 0.61 +/- 0.031
Class A Precision: 0.573 +/- 0.046
Class A Recall: 0.44 +/- 0.081
Class A F1: 0.494 +/- 0.062
PR-AUC: 0.613 +/- 0.067
